# N-CMAPSS DS05 — the two optimizations, on the representation where they can work

## Why this design

v4 established two facts that point in opposite directions:

| representation | test RMSE | Spearman(validation, test) |
|---|---|---|
| snapshot, per-row (v3) | **8.247** | **0.000** (p = 1.00) |
| cross-cycle, aggregated (v4) | 9.628 | **+0.905** (p = 0.002) |

The snapshot is more accurate and has no usable selection signal. The cross-cycle representation is
less accurate and has an almost perfect one. So this notebook runs both searches **where they can
actually work**, and treats the contrast with v3 as the experiment rather than an afterthought:
same two algorithms, same budget, two representations, one with a valid objective and one without.

Cross-cycle runs take 4-6 seconds against 693 for the snapshot, so for the first time on this
dataset a properly sized search is affordable — 40 BO trials and a 10-particle swarm, every
evaluation averaged over 3 seeds, and a two-stage confirmation on 10 fresh seeds.

## Also fixed here

The v4 ablation tested nothing. Its winner was `W = 1`, a sequence of length one, so attention over
a single timestep and max-versus-average pooling are both no-ops — R2 equalled R1 and R4 equalled
R0 to six decimal places. Part 1 re-runs those components at `W = 5`, where they are actually
active.

Window length is now a **searched hyperparameter** rather than a fixed choice, since v4 showed it
matters (9.63 at W=1 rising to 11.94 at W=20) and it costs nothing to include.

In [ ]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from scipy.stats import norm, spearmanr
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Softmax, Multiply, Lambda,
                                     GlobalAveragePooling1D, GlobalMaxPooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; SUBSAMPLE=5; VAL_UNITS=[5,6]
SNAP_TEST=8.247; SNAP_SPEARMAN=0.000        # v3 reference
set_seeds(SEED)
def read_sub(dset,step,dtype=np.float32,chunk=400_000):
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=dset[a:b]; first=(-a)%step
        if first<len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out)
S=SUBSAMPLE
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],S);  Xd=read_sub(h['X_s_dev'],S)
    Ad=read_sub(h['A_dev'],S);  Yd=read_sub(h['Y_dev'],S)
    Wt=read_sub(h['W_test'],S); Xt=read_sub(h['X_s_test'],S)
    At=read_sub(h['A_test'],S); Yt=read_sub(h['Y_test'],S)
Fd=np.hstack([Wd,Xd]).astype(np.float32); Ft=np.hstack([Wt,Xt]).astype(np.float32)
ud,cd,yd=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
ut,ct,yt=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Wd,Xd,Ad,Yd,Wt,Xt,At,Yt; gc.collect()

def aggregate(F,u,c,y):
    key=u.astype(np.int64)*100000+c; o=np.argsort(key,kind='stable')
    F,u,c,y,key=F[o],u[o],c[o],y[o],key[o]
    bnd=np.flatnonzero(np.diff(key))+1
    rows=[];meta=[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=F[a:b]; n=b-a; t=np.linspace(-1,1,n,dtype=np.float32)
        rows.append(np.concatenate([A.mean(0),A.std(0),A.min(0),A.max(0),
                                    (A*t[:,None]).sum(0)/max((t*t).sum(),1e-9)]))
        meta.append((u[a],c[a],y[a],n))
    M=np.asarray(meta,np.float64)
    return (np.asarray(rows,np.float32),M[:,0].astype(int),M[:,1].astype(int),
            M[:,2].astype(np.float32),M[:,3].astype(int))
Xc_d,uc_d,cc_d,yc_d,nr_d=aggregate(Fd,ud,cd,yd)
Xc_t,uc_t,cc_t,yc_t,nr_t=aggregate(Ft,ut,ct,yt)
TR_UNITS=[u for u in sorted(set(uc_d)) if u not in VAL_UNITS]
TRIVIAL=float(np.sqrt(mean_squared_error(yt,np.full_like(yt,yd[~np.isin(ud,VAL_UNITS)].mean()))))
print('dev %d flights | test %d flights | %d aggregated features | trivial floor %.3f'
      %(len(yc_d),len(yc_t),Xc_d.shape[1],TRIVIAL))

_CACHE={}
def build_split(W):
    if W in _CACHE: return _CACHE[W]
    def seq(Xc,uc,cc,yc,nr,units):
        X,Y,N=[],[],[]
        for uu in units:
            m=uc==uu; o=np.argsort(cc[m])
            A=np.hstack([Xc[m][o],(cc[m][o][:,None]/100.0).astype(np.float32)])  # cycle index kept
            L=yc[m][o]; R=nr[m][o]
            for e in range(W,len(A)+1): X.append(A[e-W:e]); Y.append(L[e-1]); N.append(R[e-1])
        return np.asarray(X,np.float32),np.asarray(Y,np.float32),np.asarray(N,np.int64)
    a=seq(Xc_d,uc_d,cc_d,yc_d,nr_d,TR_UNITS); b=seq(Xc_d,uc_d,cc_d,yc_d,nr_d,VAL_UNITS)
    t=seq(Xc_t,uc_t,cc_t,yc_t,nr_t,sorted(set(uc_t)))
    sc=StandardScaler().fit(a[0].reshape(-1,a[0].shape[-1]))
    f=lambda X: sc.transform(X.reshape(-1,X.shape[-1])).reshape(X.shape).astype(np.float32)
    _CACHE[W]=dict(Xtr=f(a[0]),ytr=a[1],Xva=f(b[0]),yva=b[1],nva=b[2],
                   Xte=f(t[0]),yte=t[1],nte=t[2],shape=a[0].shape[1:])
    return _CACHE[W]
def rmse_row(y,p,n):
    w=n.astype(np.float64); return float(np.sqrt(np.sum(w*(p-y)**2)/w.sum()))
for W in (1,3,5,10): 
    d=build_split(W); print('  W=%-3d train %4d | val %4d | test %4d | shape %s'
                            %(W,len(d['ytr']),len(d['yva']),len(d['yte']),d['shape']))

In [ ]:
def attn(seq,s):
    w=Softmax(axis=1)(Dense(1,activation='tanh',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=s))(seq))
    return Lambda(lambda x: K.sum(x,axis=1))(Multiply()([seq,w]))
def build(shape,seed=SEED,f1=96,f2=128,f3=32,lstm_units=80,d1=192,d2=96,dropout=0.1,
          l2reg=1e-4,lr=1e-3,bidirectional=False,attention=False,multiscale=False,pool='avg'):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=shape); c=inp
    for i,f in enumerate((f1,f2,f3)):
        k=min(3,int(c.shape[1]))
        if multiscale and int(c.shape[1])>=3:
            c=Concatenate()([Conv1D(int(f),kk,padding='same',activation='relu',**kw)(c) for kk in (2,3)])
        else:
            c=Conv1D(int(f),k,padding='same',activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<1 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=(GlobalMaxPooling1D() if pool=='max' else GlobalAveragePooling1D())(c)
    cell=LSTM(int(lstm_units),return_sequences=attention,**kw)
    r=Bidirectional(cell)(inp) if bidirectional else cell(inp)
    if attention: r=attn(r,seed)
    h=Concatenate()([cv,r])
    for dd in (d1,d2):
        h=Dense(int(dd),activation='relu',**kw)(h)
        if dropout>0: h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def run(W,seed,batch=32,epochs=400,patience=40,**kw):
    d=build_split(int(W)); K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(d['shape'],seed=seed,**kw)
        h=m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=epochs,
                batch_size=int(batch),verbose=0,
                callbacks=[EarlyStopping('val_loss',patience=patience,restore_best_weights=True),
                           ReduceLROnPlateau('val_loss',factor=0.5,patience=patience//2,min_lr=1e-6)])
        pv=m.predict(d['Xva'],batch_size=512,verbose=0).ravel()
        pt=m.predict(d['Xte'],batch_size=512,verbose=0).ravel()
    return dict(val=rmse_row(d['yva'],pv,d['nva']),test=rmse_row(d['yte'],pt,d['nte']),
                params=int(m.count_params()),secs=round(time.time()-t0,1))
def agg(W,n,seed0=SEED,**kw):
    rs=[run(W,seed0+s,**kw) for s in range(n)]
    return dict(val=float(np.mean([r['val'] for r in rs])),
                test=float(np.mean([r['test'] for r in rs])),
                test_sd=float(np.std([r['test'] for r in rs],ddof=1)) if n>1 else 0.0,
                val_sd=float(np.std([r['val'] for r in rs],ddof=1)) if n>1 else 0.0,
                params=rs[0]['params'],secs=float(np.mean([r['secs'] for r in rs])))
DEFAULTS=dict(f1=96,f2=128,f3=32,lstm_units=80,d1=192,d2=96,dropout=0.1,l2reg=1e-4,lr=1e-3)
b=agg(1,3,**DEFAULTS)
print('baseline W=1: val %.3f | test %.3f +/- %.3f | %.1f s/run'%(b['val'],b['test'],b['test_sd'],b['secs']))

## Part 1 — the ablation, corrected

Run at `W = 5` so attention and pooling actually have a sequence to act on.

In [ ]:
W_ABL=5; N=5
LAD=[('R0','baseline',dict()),('R1','+ bidirectional',dict(bidirectional=True)),
     ('R2','+ attention',dict(bidirectional=True,attention=True)),
     ('R3','+ multi-scale',dict(bidirectional=True,attention=True,multiscale=True)),
     ('R4','max pooling',dict(pool='max'))]
AB=[]
for rid,label,kw in LAD:
    a=agg(W_ABL,N,**{**DEFAULTS,**kw})
    AB.append(dict(id=rid,config=label,**a))
    print('%-3s %-18s test %6.3f +/- %5.3f | val %6.3f | params %9s'
          %(rid,label,a['test'],a['test_sd'],a['val'],format(a['params'],',')))
A=pd.DataFrame(AB); pd.set_option('display.width',200)
A['delta']=(A.test-A.test.iloc[0]).round(3)
A['noise']=np.sqrt(A.test_sd**2+A.test_sd.iloc[0]**2).round(3)
A['verdict']=np.where(A.delta.abs()>A.noise,np.where(A.delta<0,'BETTER','worse'),'within noise')
A.loc[0,'verdict']='-'
print('\n'+A[['id','config','test','test_sd','delta','noise','verdict']].to_string(index=False))
uniq=A.test.round(6).nunique()
print('\ndistinct results: %d of %d  (v4 had duplicates because W=1 made components no-ops)'%(uniq,len(A)))
r,p=spearmanr(A.val,A.test); print('Spearman(val,test) on the ablation: %+.3f (p=%.3f)'%(r,p))

## Part 2 — Bayesian optimization

Window length is searched alongside the architecture. The baseline is the first design point, so it
is in the population from the start.

In [ ]:
SPACE=[('W',1,10,'int'),('f1',32,192,'int'),('f2',32,192,'int'),('f3',16,128,'int'),
       ('lstm_units',16,192,'int'),('d1',64,320,'int'),('d2',32,192,'int'),
       ('dropout',0.0,0.5,'float'),('lr',1e-4,1e-2,'log'),('l2reg',1e-7,1e-2,'log')]
DIM=len(SPACE); BASE=dict(DEFAULTS); BASE['W']=1
BO_INIT,BO_ITERS,PSO_PART,PSO_ITERS = 8,32,10,6
TRIAL_SEEDS,CONFIRM_SEEDS,SHORTLIST = 3,10,6
def dec(v):
    kw={}
    for (n,lo,hi,k),x in zip(SPACE,np.clip(v,0,1)):
        kw[n]=int(round(lo+x*(hi-lo))) if k=='int' else (
              float(math.exp(math.log(lo)+x*(math.log(hi)-math.log(lo)))) if k=='log'
              else float(lo+x*(hi-lo)))
    return kw
def enc(kw):
    v=[]
    for n,lo,hi,k in SPACE:
        x=kw[n]
        v.append((math.log(max(x,lo))-math.log(lo))/(math.log(hi)-math.log(lo)) if k=='log' else (x-lo)/(hi-lo))
    return np.clip(np.array(v,float),0,1)
BASE_VEC=enc(BASE); LOG=[]; CACHE={}
def fit(v,tag=''):
    key=tuple(np.round(v,5))
    if key in CACHE: return CACHE[key]
    kw=dec(v); W=kw.pop('W')
    a=agg(W,TRIAL_SEEDS,**kw); CACHE[key]=a['val']
    LOG.append(dict(tag=tag,W=W,val=a['val'],test=a['test'],params=a['params'],**kw))
    print('   %-9s W=%-3d val %6.3f | test %6.3f'%(tag,W,a['val'],a['test']))
    return a['val']
print('budget: %d BO + %d PSO evaluations x %d seeds at ~%.1f s/run -> ~%.0f min'
      %(BO_INIT+BO_ITERS+1,PSO_PART*(1+PSO_ITERS),TRIAL_SEEDS,b['secs'],
        (BO_INIT+BO_ITERS+1+PSO_PART*(1+PSO_ITERS))*TRIAL_SEEDS*b['secs']/60))
BASE_VAL=fit(BASE_VEC,'baseline')

def gp_bo(n_init,n_iter,seed):
    r=np.random.RandomState(seed)
    X=[BASE_VEC.copy()]+list(r.rand(n_init,DIM))
    y=[BASE_VAL]+[fit(x,'bo-init') for x in X[1:]]
    for it in range(n_iter):
        gp=GaussianProcessRegressor(
            kernel=ConstantKernel(1.0)*Matern(length_scale=np.ones(DIM),nu=2.5)+WhiteKernel(1e-2),
            normalize_y=True,n_restarts_optimizer=2,random_state=seed)
        gp.fit(np.array(X),np.array(y))
        cand=np.random.RandomState(seed+it).rand(4000,DIM)
        mu,sd=gp.predict(cand,return_std=True); bst=min(y)
        z=(bst-mu)/np.maximum(sd,1e-9); ei=(bst-mu)*norm.cdf(z)+sd*norm.pdf(z)
        x=cand[int(np.argmax(ei))]; X.append(x); y.append(fit(x,'bo'))
        if (it+1)%8==0: print('  BO %2d: best %.3f'%(it+1,min(y)))
    return np.array(X),np.array(y)
print('\n=== Bayesian optimization ===')
t0=time.time(); BX,BY=gp_bo(BO_INIT,BO_ITERS,SEED)
print('BO best %.3f in %.0f s'%(BY.min(),time.time()-t0))

## Part 3 — particle swarm

The baseline is particle 0, so the swarm starts with it in the population.

In [ ]:
def pso(n_part,iters,seed,w=0.72,c1=1.5,c2=1.5):
    r=np.random.RandomState(seed)
    X=r.rand(n_part,DIM); X[0]=BASE_VEC.copy()
    V=r.uniform(-0.1,0.1,(n_part,DIM))
    f=np.array([fit(x,'pso-init') for x in X]); P=X.copy(); Pf=f.copy()
    G=P[int(np.argmin(Pf))].copy(); Gf=float(Pf.min())
    print('  init best %.3f (baseline particle %.3f)'%(Gf,Pf[0]))
    for it in range(iters):
        r1,r2=r.rand(n_part,DIM),r.rand(n_part,DIM)
        V=np.clip(w*V+c1*r1*(P-X)+c2*r2*(G-X),-0.25,0.25); X=np.clip(X+V,0,1)
        f=np.array([fit(x,'pso') for x in X])
        imp=f<Pf; P[imp]=X[imp]; Pf[imp]=f[imp]
        if Pf.min()<Gf: Gf=float(Pf.min()); G=P[int(np.argmin(Pf))].copy()
        print('  PSO %d: best %.3f'%(it+1,Gf))
    return G,Gf
print('=== particle swarm ===')
t0=time.time(); PG,PGf=pso(PSO_PART,PSO_ITERS,SEED); print('PSO best %.3f in %.0f s'%(PGf,time.time()-t0))
L=pd.DataFrame(LOG); L.to_csv('ncmapss_opt_log.csv',index=False)
r,p=spearmanr(L.val,L.test)
print('\nSpearman(objective, test) across %d evaluated configurations: %+.3f (p=%.4g)'%(len(L),r,p))
print('  the same statistic on the v3 snapshot representation was %.3f (p=1.00)'%SNAP_SPEARMAN)

## Part 4 — two-stage decision, and the comparison that is the point

In [ ]:
short=L.nsmallest(SHORTLIST,'val')
cands=[('baseline',BASE)]+[('cfg %d'%i,{**{n:row[n] for n,_,_,_ in SPACE if n!='W'},'W':int(row['W'])})
                           for i,row in enumerate(short.to_dict('records'))]
S2=[]
print('re-scoring %d configurations on %d fresh seeds each\n'%(len(cands),CONFIRM_SEEDS))
for nm,kw in cands:
    kw=dict(kw); W=kw.pop('W')
    a=agg(W,CONFIRM_SEEDS,seed0=SEED+900,**kw)
    S2.append(dict(name=nm,W=W,**a))
    print('  %-10s W=%-3d val %6.3f +/- %5.3f | test %6.3f +/- %5.3f'
          %(nm,W,a['val'],a['val_sd'],a['test'],a['test_sd']))
T=pd.DataFrame(S2)
base=T[T.name=='baseline'].iloc[0]; win=T.nsmallest(1,'val').iloc[0]
d=win['test']-base['test']; nz=math.sqrt(win['test_sd']**2+base['test_sd']**2)
print('\nselected on validation: %s | test %.3f vs baseline %.3f -> %+.3f against noise %.3f'
      %(win['name'],win['test'],base['test'],d,nz))
adopt = d < -nz
print('DECISION: %s'%('adopt %s'%win['name'] if adopt else 'tie - keep the baseline'))
print('\n'+'='*76)
print('%-42s %8s %10s'%('','test','Spearman'))
print('%-42s %8.3f %10.3f'%('v3 snapshot, per-row',SNAP_TEST,SNAP_SPEARMAN))
print('%-42s %8.3f %10.3f'%('cross-cycle baseline',base['test'],r))
print('%-42s %8.3f %10s'%('cross-cycle after BO and PSO',win['test'],''))
print('%-42s %8.3f %10s'%('trivial floor',TRIVIAL,''))
print('='*76)
gain=base['test']-win['test']
print('\nsearch moved the cross-cycle result by %+.3f'%(-gain))
print('on the snapshot representation, with Spearman 0.000, four searches moved it by 0.000')
print('best comparable published entry: MSTSDN 3.95 | remaining gap %+.2f'%(min(win['test'],SNAP_TEST)-3.95))
T.to_csv('ncmapss_opt_twostage.csv',index=False)
json.dump(dict(ablation=AB,log=LOG,twostage=T.to_dict('records'),spearman=[float(r),float(p)],
               snapshot=SNAP_TEST,trivial=TRIVIAL,adopt=bool(adopt)),
          open('ncmapss_optimizations.json','w'),indent=2,default=str)
print('\nsaved ncmapss_opt_log.csv, ncmapss_opt_twostage.csv, ncmapss_optimizations.json')